# Sleep-EDF: entrada para el encoder de época

Primer paso del extractor: verificar la representación tiempo-frecuencia. No instala paquetes, entrena, modifica los NPZ ni conecta prototipos. Usa NumPy, SciPy y matplotlib disponibles en Colab. Este notebook es independiente: no necesita subir `sleep_edf_dataset.py`.

El notebook de preprocesamiento del repositorio guarda `epochs.get_data()` de MNE, sin filtrado/normalización explícitos, lo que indica voltios para sus señales EEG/EOG. Comprueba que tus archivos se generaron con esa versión: en caso contrario ajusta `INPUT_UNITS`. La transformación no añade filtrado ni normalización. No demuestra todavía equivalencia con un checkpoint preentrenado.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Selección pequeña

Se leen dos archivos y se conservan solo dos épocas de cada uno. Un NPZ comprimido necesita descomprimir la matriz X de cada archivo seleccionado: el pico de memoria todavía incluye un registro completo, aunque no se conserva todo el conjunto. Puedes elegir los nombres explícitamente. No se forman secuencias entre épocas.


In [ ]:
from pathlib import Path
import numpy as np

DATA_DIR = Path('/content/drive/MyDrive/SleepCassette_procesado')
FILE_NAMES = []  # Opcional: dos nombres .npz distintos
INPUT_UNITS = 'V'  # 'V' si son voltios; 'uV' si ya son microvoltios
EPOCHS_PER_FILE = 2
assert INPUT_UNITS in ('V', 'uV')
assert DATA_DIR.is_dir(), f'No existe {DATA_DIR}'
if FILE_NAMES:
    assert len(FILE_NAMES) == 2 and len(set(FILE_NAMES)) == 2
    assert all(Path(n).name == n and n.endswith('.npz') for n in FILE_NAMES)
    selected = sorted(DATA_DIR / n for n in FILE_NAMES)
else:
    selected = sorted(DATA_DIR.glob('*.npz'))[:2]
assert len(selected) == 2 and all(p.is_file() for p in selected)
print('Seleccionados:', [p.name for p in selected])


In [ ]:
import gc

signals, labels, subjects, sample_ids = [], [], [], []
for path in selected:
    with np.load(path, allow_pickle=False) as data:
        x, y, subj = data['X'], data['y'], data['subj']
        assert x.ndim == 3 and x.shape[1:] == (3, 3000) and len(x) > 0
        assert y.shape == (len(x),) and np.issubdtype(y.dtype, np.integer)
        assert np.isin(y, np.arange(1, 7)).all()
        n = min(EPOCHS_PER_FILE, len(x))
        if subj.size == 1:
            ids = np.repeat(subj.reshape(-1), n).astype(str)
        else:
            assert subj.shape == (len(x),)
            ids = subj[:n].astype(str)
        signals.append(x[:n].astype(np.float32, copy=True))
        labels.append(y[:n].astype(np.int64) - 1)
        subjects.extend(ids.tolist())
        sample_ids.extend(f'{path.name}:{i}' for i in range(n))
    del x, y, subj
    gc.collect()
x_original = np.concatenate(signals)
y_batch = np.concatenate(labels)
del signals, labels
assert np.isfinite(x_original).all()
# Cambio de unidades explícito; no modifica los archivos ni normaliza por época.
x_uv = x_original * (1e6 if INPUT_UNITS == 'V' else 1.0)
assert np.isfinite(x_uv).all()
print('Lote:', x_uv.shape, '| Etiquetas 0–5:', y_batch)
print('Sujetos:', subjects)
print('Percentiles de amplitud absoluta por canal (microvoltios):')
print(np.percentile(np.abs(x_uv), [50, 95, 99], axis=(0, 2)))


## Espectrograma por canal

Se sigue el cálculo de `XSleepNetSpectrogram` de PhysioEx 2.0.2: ventana Hamming periódica de 200 muestras, solapamiento 100, FFT 256 y `10*log10(PSD + eps_float32)`. Se explicitan los valores por defecto de SciPy: detrend constante y escala de densidad. No se aplica clipping. Se conserva el espectrograma logarítmico sin z-score adicional.

Esto reproduce la operación espectral sobre la señal suministrada; no el preset completo `seqsleepnet`, que también filtra. La escala es dB respecto a 1 µV²/Hz, con el epsilon numérico indicado. No se puede comparar directamente con espectros calculados en voltios sin revisar escala y epsilon.


In [ ]:
from scipy.signal import spectrogram, get_window

def epoch_spectrogram(x):
    """(B, 3, 3000), en microvoltios → (B, 3, 29, 129)."""
    x = np.asarray(x, dtype=np.float64)
    assert x.ndim == 3 and x.shape[1:] == (3, 3000)
    frequencies, times, power = spectrogram(
        x, fs=100.0, window=get_window('hamming', 200),
        nperseg=200, noverlap=100, nfft=256,
        detrend='constant', scaling='density', mode='psd', axis=-1,
    )
    log_power = 10.0 * np.log10(power + np.finfo(np.float32).eps)
    specs = log_power.swapaxes(-1, -2).astype(np.float32)
    assert specs.shape == (len(x), 3, 29, 129)
    assert np.isfinite(specs).all()
    return specs, frequencies, times

specs, frequencies, times = epoch_spectrogram(x_uv)
# Cada canal será un ejemplo independiente para el mismo encoder compartido.
encoder_input = specs.reshape(-1, 1, 29, 129)
np.testing.assert_array_equal(encoder_input[0, 0], specs[0, 0])
np.testing.assert_array_equal(encoder_input[2, 0], specs[0, 2])
print('Espectrogramas:', specs.shape)
print('Entrada futura del encoder:', encoder_input.shape)
print('Frecuencias:', frequencies[0], 'a', frequencies[-1], 'Hz')


In [ ]:
import matplotlib.pyplot as plt

channel_names = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal']
time_signal = np.arange(3000) / 100.0
fig, axes = plt.subplots(3, 2, figsize=(13, 9), constrained_layout=True)
for c, name in enumerate(channel_names):
    axes[c, 0].plot(time_signal, x_uv[0, c], linewidth=0.6)
    axes[c, 0].set(title=name, xlabel='Tiempo (s)', ylabel='µV')
    image = axes[c, 1].pcolormesh(times, frequencies, specs[0, c].T, shading='auto')
    axes[c, 1].set(title=f'Espectrograma: {name}', xlabel='Tiempo (s)', ylabel='Frecuencia (Hz)')
    fig.colorbar(image, ax=axes[c, 1], label='dB re 1 µV²/Hz (con epsilon)')
plt.show()


## Qué comprobar antes de continuar

- Las señales deben tener una escala coherente con tus unidades y conservar el orden de canales. La gráfica no demuestra por sí sola que sean los canales correctos.
- No debe haber NaN ni infinitos; las dimensiones deben ser `(B, 3, 29, 129)`.
- Las etiquetas y sujetos se conservan asociados al lote. El espectrograma no los utiliza para transformar las señales.
- La entrada del encoder es `(B·3, 1, 29, 129)`. Todavía no hay características aprendidas: después incorporaremos banco de filtros, LSTM y atención para obtener `(B, 128, 1, 3)`.
- Si se busca transferencia de pesos, queda contrastar filtros, unidades y normalización con el checkpoint elegido. No volver a filtrar automáticamente ni asumir compatibilidad por las dimensiones.

Fuentes: [encoder de época](https://github.com/guidogagl/physioex/blob/v2.0.2/physioex/models/protosleepnet.py), [operación espectral](https://github.com/guidogagl/physioex/blob/v2.0.2/physioex/data/steps/spectrogram.py).

El notebook se valida localmente con señales sintéticas; las señales privadas solo se comprueban cuando lo ejecutes en Colab.
